# 02 — Le etichette MBFC sulle nostre testate (tentativo 10)

Le schede pulite (`01`) si abbinano alle testate dello scraping del 13 e 18 ottobre, in quattro insiemi annidati:
le **3.128 tentate**, quelle **con articoli scaricati**, le **863 del dataset finale** (tentativi 06–09) e le **301
valutate** (tentativo 09, gruppi reti ∪ owner ∪ Parent Domain).

- **Un dominio, una riga**: se MBFC ha più schede vive per lo stesso dominio (per esempio più TV che usano lo stesso
  sito), per ogni campo vale il valore comune; se le schede non sono d'accordo vale quello della maggioranza, e a
  parità il campo resta vuoto (`schede_in_conflitto`). Le schede di una sezione (`dominio/percorso`) non si usano.
- **Abbinamento**: prima il dominio esatto (minuscole, senza `www.`); se manca, il dominio registrato della testata
  (`news.sito.com` → `sito.com`) se MBFC ha una scheda proprio per quello.
- **Confronto con NewsGuard (agosto 2022)**: livelli l1–l5 dal punteggio, come nei tentativi 05–09; "affidabile" =
  punteggio ≥ 60. Dalla parte MBFC: credibilità alta o media, e factual reporting almeno *Mostly Factual*.
- **Controllo incrociato** con il file pubblico dell'estensione MBFC (22 ottobre): le due fonti dicono la stessa cosa?

Uscite: `Dataset/testate_mbfc.csv` (una riga per testata tentata, con le etichette MBFC e NewsGuard) e, in
`Results/1-etichette`, copertura, incroci, accordo, disaccordi e controllo incrociato.

In [ ]:
import json

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from comune import (ARTICOLI, CATEGORIA, CREDIBILITA, DATASET, FACTUAL, FINALE, MBFC_ESTENSIONE, NEWSGUARD, ORIENTAMENTO,
                    RESULTS, TENTATE, livello_newsguard, registrato)

pd.set_option("display.width", 200)
LIV = ["l1", "l2", "l3", "l4", "l5"]
m = pd.read_csv(DATASET / "mbfc_pulito.csv")
m = m[~m.morto & m.percorso.isna() & m.dominio.notna()]
print(len(m), "schede vive di un sito intero,", m.dominio.nunique(), "domini")

## Un dominio, una riga

In [ ]:
CAMPI = ["categoria", "orientamento", "factual", "factual_score", "credibilita", "paese", "tipo_media"]


def maggioranza(x):
    c = x.dropna().value_counts()
    if c.empty:
        return np.nan
    return c.index[0] if (c == c.iloc[0]).sum() == 1 else np.nan


righe = []
for dom, g in m.groupby("dominio"):
    r = {"dominio": dom, "schede_mbfc": len(g), "nome_mbfc": " | ".join(g.nome.astype(str)), "url_mbfc": g.url_mbfc.iloc[0],
         "id_mbfc": g.id_mbfc.iloc[0], "score_incoerente": bool(g.score_incoerente.any())}
    conflitti = [c for c in CAMPI if g[c].dropna().nunique() > 1]
    for c in CAMPI:
        r[c] = g[c].iloc[0] if len(g) == 1 else maggioranza(g[c])
    r["schede_in_conflitto"] = ", ".join(conflitti)
    righe.append(r)
dom = pd.DataFrame(righe).set_index("dominio")
print(f"{len(dom)} domini; {int((dom.schede_mbfc > 1).sum())} con più schede, di cui {int((dom.schede_in_conflitto != '').sum())} in disaccordo su qualche campo")
dom[dom.schede_in_conflitto != ""][["nome_mbfc", "schede_in_conflitto", "factual", "credibilita"]].head(15)

## Le nostre testate e l'abbinamento

In [ ]:
tentate = pd.read_csv(TENTATE).dominio.str.lower().str.removeprefix("www.").unique()
con_articoli = set(pd.read_csv(ARTICOLI, usecols=["dominio"]).dominio.str.lower())
fin = pd.read_csv(FINALE).set_index("domain")
ng = pd.read_csv(NEWSGUARD, low_memory=False)
ng = ng.assign(d=ng.Domain.str.lower()).drop_duplicates("d").set_index("d")
t = pd.DataFrame(index=pd.Index(sorted(set(tentate) | set(fin.index)), name="domain"))
t["con_articoli"] = t.index.isin(con_articoli)
t["dataset_finale"] = t.index.isin(fin.index)
t["valutata"] = t.index.map(fin.valutazione).fillna(False).astype(bool)
t["gruppo"] = t.index.map(fin.gruppo); t["rete"] = t.index.map(fin.rete)
t["ng_score"] = t.index.map(ng.Score)
t["ng_livello"] = livello_newsguard(t.ng_score.to_numpy()).to_numpy()
t["ng_aggiornato"] = pd.to_datetime(t.index.map(ng["Last Updated"]), unit="ms").date


def abbina(d):
    if d in dom.index:
        return d, "dominio esatto"
    r = registrato(d)
    if r != d and r in dom.index:
        return r, "dominio registrato"
    return np.nan, np.nan


ab = [abbina(d) for d in t.index]
t["dominio_mbfc"] = [a for a, _ in ab]; t["abbinamento"] = [b for _, b in ab]
t = t.join(dom, on="dominio_mbfc")
t["mbfc"] = t.dominio_mbfc.notna()
t.to_csv(DATASET / "testate_mbfc.csv")
print(t.abbinamento.value_counts().to_dict())

## Quante restano fuori

In [ ]:
INSIEMI = {"tentate nello scraping": t.index == t.index, "con articoli scaricati": t.con_articoli,
           "dataset finale": t.dataset_finale, "valutate": t.valutata}
righe = []
for nome, sel in INSIEMI.items():
    x = t[sel]
    r = {"insieme": nome, "testate": len(x), "in MBFC": int(x.mbfc.sum()), "quota": x.mbfc.mean(),
         "dominio esatto": int((x.abbinamento == "dominio esatto").sum()), "dominio registrato": int((x.abbinamento == "dominio registrato").sum())}
    for k, l in enumerate(LIV):
        y = x[x.ng_livello == k]; r[f"{l} (in MBFC/tutte)"] = f"{int(y.mbfc.sum())}/{len(y)}"
    righe.append(r)
cop = pd.DataFrame(righe).set_index("insieme"); cop.to_csv(RESULTS / "02_copertura.csv")
nn = t.dataset_finale & (t.rete == 0)
print(f"rete Natural News nel dataset finale: {int(nn.sum())} testate, {int((nn & t.mbfc).sum())} in MBFC")
cop.round(3)

## Come si distribuiscono factual, credibilità e bias, per livello NewsGuard

In [ ]:
ORDINI = {"factual": FACTUAL, "credibilita": CREDIBILITA, "orientamento": ORIENTAMENTO, "categoria": CATEGORIA}
incroci = []
for nome, sel in [("dataset finale", t.dataset_finale), ("valutate", t.valutata)]:
    x = t[sel & t.mbfc]
    for c, ordine in ORDINI.items():
        tab = pd.crosstab(x[c].fillna("(vuoto)"), x.ng_livello.map(dict(enumerate(LIV))))
        tab = tab.reindex([o for o in ordine + ["(vuoto)"] if o in tab.index]).reindex(columns=LIV, fill_value=0)
        tab["tutte"] = tab.sum(1)
        incroci.append(tab.assign(insieme=nome, campo=c).reset_index(names="valore"))
        if nome == "dataset finale":
            print(f"\n== {c} (MBFC) per livello NewsGuard, {nome}: {len(x)} testate"); print(tab.to_string())
pd.concat(incroci).to_csv(RESULTS / "02_incroci_per_livello_newsguard.csv", index=False)

## Accordo con NewsGuard

Sul binario: NewsGuard affidabile (≥ 60) contro MBFC credibilità alta/media, e contro factual almeno *Mostly
Factual*. Sull'ordine: correlazione di Spearman tra il punteggio NewsGuard e il factual score MBFC (0 = migliore,
quindi ci si aspetta una correlazione negativa). Anche con una testata per gruppo, perché le reti contano molte volte
la stessa valutazione.

In [ ]:
righe = []
for nome, x in [("dataset finale", t[t.dataset_finale & t.mbfc]), ("valutate", t[t.valutata & t.mbfc]),
                ("valutate, una per gruppo", t[t.valutata & t.mbfc].drop_duplicates("gruppo")),
                ("tentate con punteggio NewsGuard", t[t.mbfc & t.ng_score.notna()])]:
    x = x[x.ng_score.notna()]
    ng_si = x.ng_score >= 60
    for etichetta, campo, mb in [("credibilità alta/media", "credibilita", x.credibilita.isin(["High", "Medium"])),
                                 ("factual almeno Mostly Factual", "factual", x.factual.isin(["Very High", "High", "Mostly Factual"]))]:
        ok = x[campo].notna()
        righe.append({"insieme": nome, "MBFC": etichetta, "testate": int(ok.sum()), "accordo": (ng_si == mb)[ok].mean(),
                      "NG sì, MBFC no": int((ng_si & ~mb)[ok].sum()), "NG no, MBFC sì": int((~ng_si & mb)[ok].sum())})
    ok = x.factual_score.notna()
    rho = spearmanr(x.ng_score[ok], x.factual_score[ok]).statistic
    righe.append({"insieme": nome, "MBFC": "Spearman punteggio NG ~ factual score", "testate": int(ok.sum()), "accordo": rho})
acc = pd.DataFrame(righe); acc.to_csv(RESULTS / "02_accordo_newsguard.csv", index=False)
acc.round(3)

In [ ]:
x = t[t.dataset_finale & t.mbfc & t.ng_score.notna()]
ng_si = x.ng_score >= 60; mb = x.credibilita.isin(["High", "Medium"])
dis = x[(ng_si != mb) & x.credibilita.notna()][["valutata", "ng_score", "ng_livello", "ng_aggiornato", "nome_mbfc", "categoria",
                                              "orientamento", "factual", "credibilita", "url_mbfc"]]
dis = dis.assign(verso=np.where(dis.ng_score >= 60, "NG affidabile, MBFC bassa credibilità", "NG inaffidabile, MBFC alta/media"))
dis.sort_values(["verso", "ng_score"]).to_csv(RESULTS / "02_disaccordi_newsguard.csv")
dis.sort_values(["verso", "ng_score"])[["verso", "ng_score", "nome_mbfc", "categoria", "orientamento", "factual", "credibilita"]]

## Controllo incrociato: API contro estensione MBFC

In [ ]:
e = json.load(open(MBFC_ESTENSIONE))
es = pd.DataFrame(e["sources"]).set_index("domain")
mappa_f = {"very-high": "Very High", "high": "High", "mostly-factual": "Mostly Factual", "mixed": "Mixed", "low": "Low", "very-low": "Very Low"}
mappa_c = {"high-credibility": "High", "medium-credibility": "Medium", "low-credibility": "Low"}
x = t[t.mbfc & t.dominio_mbfc.isin(es.index)]
f_e = x.dominio_mbfc.map(es.reporting).map(mappa_f); c_e = x.dominio_mbfc.map(es.credibility).map(mappa_c)
ctrl = pd.DataFrame([{"campo": "factual", "testate": int((x.factual.notna() & f_e.notna()).sum()), "uguali": (x.factual == f_e)[x.factual.notna() & f_e.notna()].mean()},
                     {"campo": "credibilità", "testate": int((x.credibilita.notna() & c_e.notna()).sum()), "uguali": (x.credibilita == c_e)[x.credibilita.notna() & c_e.notna()].mean()}])
ctrl["testate abbinate solo nell'API"] = int((t.mbfc & ~t.dominio_mbfc.isin(es.index)).sum())
ctrl.to_csv(RESULTS / "02_controllo_estensione.csv", index=False)
diversi = x[(x.factual != f_e) & x.factual.notna() & f_e.notna()]
print("factual diverso tra API ed estensione:", diversi.assign(estensione=f_e[diversi.index])[["nome_mbfc", "factual"]].assign(estensione=f_e[diversi.index]).head(10).to_string())
ctrl.round(3)